In [1]:

import sqlite3
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# --- 1. DATA EXTRACTION ---
# Connect to SQLite (Update 'db.sqlite3' and 'yourapp_' prefixes to match your Django app)
conn = sqlite3.connect('db.sqlite3')

query = """
SELECT 
    eps.id as sizing_id,
    e.name as edge_name,
    n_start.name as source_node,
    n_end.name as target_node,
    eps.peak_load_kw,
    eps.mass_flow_kg_s,
    eps.diameter_m,
    eps.diameter_m * 1000 as diameter_mm,
    eps.velocity_m_s,
    eps.pressure_drop_pa,
    e.length
FROM api_pipesizing ps
JOIN api_edgepipesizing eps ON ps.id = eps.pipe_sizing_id
JOIN api_edge e ON eps.edge_id = e.id
JOIN api_node n_start ON e.start_node_id = n_start.id
JOIN api_node n_end ON e.end_node_id = n_end.id
WHERE ps.id = 2
"""
df = pd.read_sql_query(query, conn)
conn.close()
print("Data extracted successfully. Sample:")
display(df.head())
# --- 2. ENGINEERING METRICS DERIVATION ---
# Calculate Friction Gradient (Pressure drop per meter) - the ultimate metric for pipe stress
df['friction_gradient_pa_m'] = df['pressure_drop_pa'] / df['length']

# Calculate Total Head Loss in kPa for cleaner reading
df['head_loss_kpa'] = df['pressure_drop_pa'] / 1000.0

# Flag structural limits
MAX_VELOCITY = 2.5
TARGET_FRICTION = 100

display(df[['edge_name', 'peak_load_kw', 'diameter_mm', 'velocity_m_s', 'friction_gradient_pa_m']].head())

# %% [markdown]
# ### Visualization 1: Thermal Load Topology (Sankey Diagram)
# **The Macro View:** Visualizes the macro-distribution of thermal power (kW). This instantly highlights which main trunk lines are carrying the most critical loads before splitting into the capillary network.

# %%
all_nodes = list(pd.unique(df[['source_node', 'target_node']].values.ravel('K')))
node_dict = {node: i for i, node in enumerate(all_nodes)}

fig_sankey = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node = dict(
      pad = 25,
      thickness = 20,
      line = dict(color = "#111", width = 1),
      label = all_nodes,
      color = "#2b2b2b"
    ),
    link = dict(
      source = df['source_node'].map(node_dict).tolist(),
      target = df['target_node'].map(node_dict).tolist(),
      value = df['peak_load_kw'].tolist(),
      hovertemplate='<b>Link:</b> %{source.label} → %{target.label}<br><b>Thermal Load:</b> %{value:,.1f} kW<extra></extra>',
      color = "rgba(0, 214, 143, 0.5)" # Fluid Green
    )
)])

fig_sankey.update_layout(
    title_text="Thermal Load Cascade (kW)", 
    font_size=12, 
    height=600,
    template="plotly_white"
)
fig_sankey.show()

# %% [markdown]
# ### Visualization 2: Hydraulic Stress Matrix (Scatter)
# **The Diagnostic View:** This isolates exactly where the physics break down. We plot Velocity vs. Friction Gradient. 
# * **Top Right:** The Danger Zone. High velocity (erosion risk) and high friction (pump starvation). 
# * **Bottom Left:** Stagnant flow. Risk of temperature loss and air entrainment.

# %%
fig_matrix = px.scatter(
    df, 
    x="velocity_m_s", 
    y="friction_gradient_pa_m", 
    size="mass_flow_kg_s", 
    color="diameter_mm",
    hover_name="edge_name",
    hover_data={"peak_load_kw": ":.1f", "length": ":.1f"},
    color_continuous_scale=px.colors.sequential.Sunsetdark,
    title="Hydraulic Stress Matrix: Velocity vs. Friction Gradient",
    labels={
        "velocity_m_s": "Fluid Velocity (m/s)",
        "friction_gradient_pa_m": "Friction Gradient (Pa/m)",
        "mass_flow_kg_s": "Mass Flow (kg/s)",
        "diameter_mm": "Size (DN)"
    }
)

# Overlay Engineering Constraints
fig_matrix.add_vline(x=MAX_VELOCITY, line_width=2, line_dash="dash", line_color="#ff3d71", annotation_text="Erosion Limit (2.5 m/s)")
fig_matrix.add_hline(y=TARGET_FRICTION, line_width=2, line_dash="dash", line_color="#ffaa00", annotation_text="Design Target (100 Pa/m)")

# Highlight Danger Zone
fig_matrix.add_vrect(x0=MAX_VELOCITY, x1=df['velocity_m_s'].max() * 1.1, fillcolor="#ff3d71", opacity=0.1, layer="below")

fig_matrix.update_layout(height=600, template="plotly_white")
fig_matrix.show()

# %% [markdown]
# ### Visualization 3: Pressure Drop Pareto Analysis (Bar)
# **The Systems View:** Which specific pipe segments are forcing us to buy a larger pump? This chart sorts total absolute pressure drop (in kPa) by segment, color-coded by how fast the water is moving.

# %%
# Sort by highest absolute pressure drop
df_sorted = df.sort_values('head_loss_kpa', ascending=False)

fig_pareto = px.bar(
    df_sorted, 
    x="edge_name", 
    y="head_loss_kpa",
    color="velocity_m_s",
    title="System Head Loss by Segment (Choke Point Identification)",
    color_continuous_scale=px.colors.diverging.Tealrose,
    color_continuous_midpoint=1.5, # Midpoint of optimal velocity
    labels={
        "edge_name": "Pipe Segment",
        "head_loss_kpa": "Total Head Loss (kPa)",
        "velocity_m_s": "Velocity (m/s)"
    }
)

fig_pareto.update_layout(
    height=500, 
    xaxis_tickangle=-45,
    template="plotly_white",
    xaxis_title="Network Branches (Sorted by Impact)"
)
fig_pareto.show()

# %% [markdown]
# ### Visualization 4: Multidimensional Parallel Coordinates
# **The "God-Mode" View:** How do all our variables interact? This allows you to trace a single pipe segment through its Peak Demand, Size, Velocity, and resulting Pressure Drop simultaneously to spot systemic sizing anomalies.

# %%
fig_parallel = px.parallel_coordinates(
    df, 
    color="diameter_mm",
    dimensions=["length", "peak_load_kw", "mass_flow_kg_s", "diameter_mm", "velocity_m_s", "friction_gradient_pa_m"],
    color_continuous_scale=px.colors.diverging.Portland,
    title="Systemic Parameter Tracing (Parallel Coordinates)",
    labels={
        "length": "Length (m)",
        "peak_load_kw": "Demand (kW)",
        "mass_flow_kg_s": "Flow (kg/s)",
        "diameter_mm": "Size (DN)",
        "velocity_m_s": "Velocity (m/s)",
        "friction_gradient_pa_m": "Friction (Pa/m)"
    }
)

fig_parallel.update_layout(height=500)
fig_parallel.show()

Data extracted successfully. Sample:


,sizing_id,edge_name,source_node,target_node,peak_load_kw,mass_flow_kg_s,diameter_m,diameter_mm,velocity_m_s,pressure_drop_pa,length
0,476,Data Center-BLDG023965-0-1,Data Center,BLDG023965,3455.31,11.780805,0.08,80.0,2.411727,259933.093231,406.253513
1,477,BLDG023965-BLDG023964-1-2,BLDG023965,BLDG023964,3438.77,11.724412,0.08,80.0,2.400183,7050.330511,11.122957
2,478,BLDG023964-BLDG023962-2-4,BLDG023964,BLDG023962,11.83,0.040334,0.02,20.0,0.132113,373.761164,23.751642
3,479,BLDG023964-BLDG023963-2-3,BLDG023964,BLDG023963,3414.37,11.641221,0.08,80.0,2.383152,5779.490431,9.245863
4,480,BLDG023962-BLDG029477-4-19,BLDG023962,BLDG029477,9.03,0.030788,0.02,20.0,0.100844,2737.897602,279.930349


,edge_name,peak_load_kw,diameter_mm,velocity_m_s,friction_gradient_pa_m
0,Data Center-BLDG023965-0-1,3455.31,80.0,2.411727,639.829774
1,BLDG023965-BLDG023964-1-2,3438.77,80.0,2.400183,633.853994
2,BLDG023964-BLDG023962-2-4,11.83,20.0,0.132113,15.736224
3,BLDG023964-BLDG023963-2-3,3414.37,80.0,2.383152,625.089349
4,BLDG023962-BLDG029477-4-19,9.03,20.0,0.100844,9.780639
